In [3]:
from performance import psnr, fsi, ssim
from zigzag import zigzag, inverse_zigzag
from PIL import Image
from pathlib import Path
import cv2
import numpy as np
import import_ipynb
import jpeglib
import random

In [4]:
np.set_printoptions(suppress=True)

In [25]:
def convert_other_format_to_jpeg(other_path, jpeg_path, quality=100):
    with Image.open(other_path) as img:
        rgb_img = img.convert('RGB')  # Convert to RGB if it's not already
        # rgb_img.save(jpeg_path, 'JPEG', quality=quality, subsampling=0, optimize=False)
        # gray_img = img.convert('L')  # Convert to grayscale
        image_jpeg = rgb_img.resize((512, 512))  # Resize to 512x512
        image_jpeg.save(jpeg_path, 'JPEG', quality=quality, subsampling=0, optimize=False)
    print(f"Converted {other_path} to {jpeg_path} with quality {quality}")

In [ ]:
# baboon_path = "cover-images/misc/4.2.03.tiff"
# peppers_path = "cover-images/misc/4.2.07.tiff"
# convert_other_format_to_jpeg(baboon_path, "cover-images/baboon_rgb.jpg", quality=100)
# convert_other_format_to_jpeg(peppers_path, "cover-images/peppers_rgb.jpg", quality=100)

Converted cover-images/misc/4.2.03.tiff to cover-images/baboon_rgb.jpg with quality 100
Converted cover-images/misc/4.2.07.tiff to cover-images/peppers_rgb.jpg with quality 100


In [20]:
bossbase_folder = f"C:/Users/LENOVO/Downloads/archive/GBRASNET/BOSSbase-1.01/cover"
path = Path(bossbase_folder)

def get_all_bossbase_files():
    files = [f for f in path.iterdir() if f.is_file()]
    base_list = []
    for idx, file in enumerate(files):
        components = list(file.parts)
        pgm_name = components[-1]
        base_name = pgm_name.split('.')[0]
        base_list.append(base_name)
    random_file = random.choices(base_list, k=1100)
    print(f"Randomly selected file: {random_file}")
    print(f"Total files in BOSSbase: {len(files)}")
    print(f"Total randomly selected files: {len(random_file)}")
    return random_file

In [24]:
def convert_random_bossbase_to_jpeg():
    random_file = get_all_bossbase_files()
    cover_folder = "cover-images/bossbase_jpeg/"
    store_path = Path(cover_folder)
    store_path.mkdir(parents=True, exist_ok=True)
    for file in random_file:
        pgm_path = path / f"{file}.pgm"
        jpeg_path = store_path / f"{file}.jpg"
        convert_other_format_to_jpeg(pgm_path, jpeg_path, quality=100)
    store_files = [f for f in store_path.iterdir() if f.is_file()]
    print(f"Total JPEG files created: {len(store_files)}") 

In [8]:
# Quantization matrix (Q50)
base_q_mat = np.asarray([[16, 11, 10, 16,  24, 40,   51,  61],
                    [12, 12, 14, 19,  26, 58,   60,  55],
                    [14, 13, 16, 24,  40, 57,   69,  56],
                    [14, 17, 22, 29,  51, 87,   80,  62],
                    [18, 22, 37, 56,  68, 109, 103,  77],
                    [24, 36, 55, 64,  81, 104, 113,  92],
                    [49, 64, 78, 87, 103, 121, 120, 101],
                    [72, 92, 95, 98, 112, 100, 103,  99]
                  ], dtype = np.float64)

In [9]:
def custom_q_mat(Q):
    if Q < 50:
        S = 5000 / Q
    else:
        S = 200 - 2 * Q
    Ts = np.floor((base_q_mat * S + 50) / 100)
    Ts[Ts < 1] = 1
    return Ts.astype(np.uint8)

def custom_q_mat2(Q):
    if Q < 50:
        S = 5000 / Q
    else:
        S = 200 - 2 * Q
    Ts = np.floor((base_q_mat * S + 50) / 100) / 35.0
    Ts[Ts < 1] = 1
    return Ts.astype(np.uint8)

In [10]:
custom_q_mat(90)

array([[ 3,  2,  2,  3,  5,  8, 10, 12],
       [ 2,  2,  3,  4,  5, 12, 12, 11],
       [ 3,  3,  3,  5,  8, 11, 14, 11],
       [ 3,  3,  4,  6, 10, 17, 16, 12],
       [ 4,  4,  7, 11, 14, 22, 21, 15],
       [ 5,  7, 11, 13, 16, 21, 23, 18],
       [10, 13, 16, 17, 21, 24, 24, 20],
       [14, 18, 19, 20, 22, 20, 21, 20]], dtype=uint8)

In [11]:
custom_q_mat2(50)

array([[1, 1, 1, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 2, 2, 1],
       [1, 1, 1, 1, 1, 3, 2, 2],
       [1, 1, 1, 1, 2, 2, 3, 2],
       [1, 1, 2, 2, 2, 3, 3, 2],
       [2, 2, 2, 2, 3, 2, 2, 2]], dtype=uint8)

In [12]:
def change_image_QF(image_path, target_qf):
    im = jpeglib.read_dct(image_path)
    old_qt = im.qt[0]

    # From QF 100 to target QF
    dequantized = im.Y.astype(np.float64) * old_qt
    new_coefficients = np.round(dequantized / custom_q_mat(target_qf)).astype(np.int16)
    
    # Update image object
    im.Y[:] = new_coefficients
    im.qt[0] = custom_q_mat(target_qf)

    get_ext = image_path.split(".")[-1]
    if get_ext != "jpeg" and get_ext != "jpg":
        print(f"Warning: The file {image_path} is not a JPEG image.")
        return
    ori_path =  image_path.split(f".{get_ext}")[0]
    output_path = f"{ori_path}_qf{target_qf}.{get_ext}"
    im.write_dct(output_path)

In [ ]:
custom_quality = [90, 80, 70, 60, 50]
for qf in custom_quality:
    change_image_QF("cover-images/baboon_rgb.jpg", qf)
    change_image_QF("cover-images/peppers_rgb.jpg", qf)

In [14]:
def change_image_QF_high_quality(image_path, target_qf):
    im = jpeglib.read_dct(image_path)
    old_qt = im.qt[0]

    # From QF 100 to target QF
    dequantized = im.Y.astype(np.float64) * old_qt
    new_coefficients = np.round(dequantized / custom_q_mat(target_qf)).astype(np.int16)
    
    # Update image object
    im.Y[:] = new_coefficients
    im.qt[0] = custom_q_mat(target_qf)

    dequantized = im.Y.astype(np.float64) * custom_q_mat(target_qf)
    im.Y[:] = np.round(dequantized / custom_q_mat(100)).astype(np.int16)
    im.qt[0] = custom_q_mat(100)
    
    ori_path =  image_path.split(".jpeg")[0]
    output_path = f"{ori_path}_qf{target_qf}_hd.jpeg"
    im.write_dct(output_path)

In [15]:
# Transform spatial to frequency domain
def transform_to_freq(imgarr, q_mat):
    sorted_coefficients = []
    h, w = imgarr.shape
    block_size = 8
    
    for i in range(0, h, block_size):
        for j in range(0, w, block_size):
            # Partition image into 8 x 8 blocks
            block = imgarr[i:i+block_size, j:j+block_size]
            # Shift values by 128
            block_f = np.float64(block) - 128.0  
            # DCT transform
            coeff = cv2.dct(block_f) 
            # Quantization
            q_coeff = np.around((coeff / q_mat).astype(np.float64))
            # Zigzag scan
            zigzag_scan = zigzag(q_coeff)
            sorted_coefficients.append(zigzag_scan)

    return sorted_coefficients

In [16]:
def reconstruct_image(sorted_coefficients, q_mat, img_shape):
    h, w = img_shape
    block_size = 8
    stego_img = np.zeros((h, w))
    idx = 0
    for i in range(0, h, block_size):
        for j in range(0, w, block_size):
            # Inverse zigzag → matriks 8×8
            zz = sorted_coefficients[idx]
            block_q = inverse_zigzag(zz, 8, 8)
            # Dequantization
            block_deq = block_q.astype(np.float64) * q_mat
            # Inverse DCT
            block_spatial = cv2.idct(block_deq)
            # Shift back by +128
            block_spatial += 128.0
            stego_img[i:i+8, j:j+8] = block_spatial
            idx += 1

    # Normalize and convert
    return np.uint8(np.clip(np.round(stego_img), 0, 255))